# TTD Stratified Segment Selection

Selects stratified groups of segments per bucket x geo for different purposes,
built directly on top of `Price_Assign_2026-05-26.csv`, which now carries
`geo`, `geo_rank`, and `group` (segments chunked into groups of ~3 by revenue
rank within each provider x geo, restarting per provider x geo -- built
externally, not re-derived here).

- **Group 1 -- holdout**: individual segments to exclude from future price
  changes. Prefers `delta==0` and the most stable (flattest, lowest
  week-to-week volatility) candidates -- same as before -- **plus a new
  constraint: at most one selected segment per `group` number** (segments
  from the same group are correlated by construction, so the holdout
  shouldn't cluster inside a handful of groups).
- **Group 2 -- experiment 2 candidates**: operates at **group** granularity,
  not segment granularity. Selects groups (preferring groups with a
  `delta!=0` available member, lowest average stability among available
  members) from the pool with at least `min_available_members` members left
  after removing whatever Group 1's holdout already claimed. **A holdout
  pick only protects its own segment, not the whole group** -- since Group 1
  enforces at most one holdout segment per group, a group can appear in
  BOTH outputs: one member held out at its current price, the others
  reassigned here. Within each selected group, its available members are
  randomly assigned one each of the new delta values (`NEW_DELTAS`, default
  `0 / -0.025 / +0.025`) -- mirrors how the original experiment's groups were
  built (rank-adjacent segments, one control + treated arms), just reusing
  the existing group structure instead of forming new ones.

**Known data quirk**: 34% of groups span more than one revenue bucket (the
`group` field was built by pure rank-chunking, not bucket-aware). A group's
bucket for stratification purposes is its **majority bucket among members,
ties broken toward the higher bucket**.

Both output files get a `tag` column identifying their purpose, with the
segment's own new delta appended, e.g. `"experiment 2 (delta = +0.025)"` /
`"control 2 (delta = +0.000)"` / `"experiment 1 holdout (delta = +0.000)"`.

## Config

In [1]:
import pandas as pd
import numpy as np
import os
import hashlib

data_path = "/Users/ravirajan/code/data"
output_path = "/Users/ravirajan/code/outputs/TTD"

# ── universe: now carries geo/geo_rank/group directly (INPUT VARIABLE) ───────
universe_file = f"{output_path}/Price_Assign_2026-05-26.csv"
provider = "sharethis"      # dav2shreths never changes price -- not a candidate for either group

# ── weekly TTD export: source of trend/volatility (INPUT VARIABLE) ───────────
ttd_weekly_file = f"{data_path}/TTD-20260714-AllOnly.csv"
treatment_date = "2026-05-25"          # pre-period = weeks strictly before this
max_week_nan_frac = 0.5                # auto-drop weeks with >50% unmatched element ids

# ── bucket boundaries, same convention as TTD3_Pricing.ipynb (INPUT VARIABLE) ─
# 1000+ retired -- 300+ is now the top tier (everything >= 300 lumped together).
BUCKET_BOUNDARIES = [300, 100, 10, 1]

# ── shared eligibility floor across every group (INPUT VARIABLE) ─────────────
min_revenue = 0         # eligible = revenue > min_revenue (segment-level for Group 1,
                         # group max-member-revenue for Group 2)

# ── run date, used in every group's output filename (INPUT VARIABLE) ─────────
run_date = "2026-07-24"

# ── Group 1 -- holdout (INPUT VARIABLE) ───────────────────────────────────────
group1_pct = 0.10
group1_tag = "experiment 1 holdout"       # base tag; actual delta is appended per row, e.g. "(delta = +0.000)"
group1_output_file = f"{output_path}/holdout_list_{run_date}.csv"
group1_pct_overrides = {}      # {(bucket, geo): pct}
group1_min_n_overrides = {}    # {(bucket, geo): n}
group1_min_n_floor = 10        # blanket floor: at least 10 holdout segments per (bucket, geo) cell
                                # (capped at that cell's eligible pool size)

# ── Group 2 -- experiment 2 candidates (INPUT VARIABLE) ──────────────────────
# Drawn from groups EXCLUDING whatever Group 1 already claimed. New deltas are
# randomly assigned to each selected group's actual members -- this notebook
# both selects AND assigns for Group 2 (unlike Group 1, which only selects;
# holdout means "leave price as-is", so there's no new delta to assign).
group2_pct = 0.10
group2_tag = "experiment 2"               # base tag for members assigned a nonzero new_delta
group2_control_tag = "control 2"          # base tag for members assigned new_delta == 0
                                           # actual delta is appended per row, e.g. "(delta = +0.025)"
group2_output_file = f"{output_path}/experiment2_candidates_{run_date}.csv"
NEW_DELTAS = [0.0, -0.025, 0.025]

group2_pct_overrides = {
    ("100+", "Global"): 0.30,
    ("100+", "US"): 0.30,
    ("300+", "Global"): 0.30,
    ("300+", "US"): 0.30,
}
group2_min_n_overrides = {}    # blanket floor below now covers every cell uniformly
group2_min_n_floor = 5         # blanket floor: at least 5 groups per (bucket, geo) cell
                                # (capped at that cell's eligible pool size)

# ── restore_segments -- kept in sync with TTD_Price_Assign.ipynb's own list
# (the >$1000-revenue sharethis/Global segments mistakenly placed in
# experiment 1, being restored to base_PoM_default). Not a strict requirement
# that Group 2 avoid these -- toggle via exclude_restore_segments_from_group2 ─
restore_segments = [
    'Global > Validated Demographic > Age > 18-44',
    'Global > people & society > social issues & advocacy > green living & environmental issues',
    'Global > hobbies & leisure > outdoors',
    'Global > travel > tourist destinations > mountain & ski resorts',
    'Global > business & industrial > hospitality industry > food service',
    'Global > law & government > military > veterans',
    'Global > computers & electronics > networking > vpn & remote access',
    'Global > finance > investing',
    'Global > Validated Demographic > Gender and Age > Females 18-34',
    'Global > business & industrial > transportation & logistics > freight & trucking',
]
exclude_restore_segments_from_group2 = True   # config: keep restore_segments out of Group 2 too

# Some Full_Path values span more than one Third_Party_Data_Provider_Element_Id
# (delimited with "|" upstream, e.g. by TTD_Price_Assign.ipynb's segment-collapse
# fix) -- pricing one such "compound" segment as a single unit is ambiguous, since
# it's really 2+ distinct elements. Default: only single-element Full_Path values
# are eligible to be selected FOR TREATMENT (Group 2) -- holdout is unaffected,
# since holdout doesn't introduce any new price/ambiguity.
require_unique_element_id_for_treatment = True

# NOTE: Group 2's per-group delta assignment (below) now seeds its own RNG
# per group from a stable hash of that group's identity, rather than drawing
# sequentially from one shared stream -- see the assignment cell for why.
# No shared RNG needed here any more.

## Universe — bucket assignment, group-level aggregation

In [2]:
_boundaries = sorted(set(BUCKET_BOUNDARIES), reverse=True)
_labels = [f"{b:g}+" for b in _boundaries]
ZERO_LABEL = "0+"
BUCKET_ORDER = [ZERO_LABEL] + list(reversed(_labels))
BUCKET_RANK = {b: i for i, b in enumerate(BUCKET_ORDER)}   # 0 = lowest .. len-1 = highest
GEO_ORDER = ["Global", "US", "Asia", "APAC", "Unknown"]

universe = pd.read_csv(universe_file)
universe = universe[universe["Third_Party_Data_Provider_Id"] == provider].copy()
# Missing revenue = $0/mo, not "unknown" -- NaN would otherwise fail every
# revenue>=threshold bucket comparison silently and also NaN <= threshold
# comparisons elsewhere (always False in pandas), rather than being an
# explicit, visible decision.
universe["revenue"] = universe["revenue"].fillna(0)
universe["bucket"] = np.select([universe["revenue"] >= b for b in _boundaries], _labels, default=ZERO_LABEL)
universe["bucket_rank"] = universe["bucket"].map(BUCKET_RANK)

print(f"{len(universe):,} {provider} segments loaded from {universe_file}")
print(universe.groupby(["bucket", "geo"]).size().unstack(fill_value=0)
      .reindex(index=BUCKET_ORDER, columns=GEO_ORDER, fill_value=0))

3,580 sharethis segments loaded from /Users/ravirajan/code/outputs/TTD/Price_Assign_2026-05-26.csv
geo     Global   US  Asia  APAC  Unknown
bucket                                  
0+         559  498  1111     1      321
1+         184  147    85     0       91
10+        200   98    21     4       69
100+        50   49     1     7       10
300+        27   39     0     0        8


In [3]:
GROUP_KEYS = ["Third_Party_Data_Provider_Id", "geo", "group"]

def _majority_bucket_rank(sub):
    counts = sub["bucket_rank"].value_counts()
    top = counts[counts == counts.max()].index
    return max(top)   # tie -> higher bucket

_group_bucket_rank = universe.groupby(GROUP_KEYS).apply(_majority_bucket_rank, include_groups=False)
group_table = _group_bucket_rank.rename("bucket_rank").reset_index()
_rank_to_bucket = {v: k for k, v in BUCKET_RANK.items()}
group_table["bucket"] = group_table["bucket_rank"].map(_rank_to_bucket)

_group_meta = universe.groupby(GROUP_KEYS).agg(
    n_members=("delta", "size"),
    any_nonzero=("delta", lambda s: (s != 0).any()),
    max_revenue=("revenue", "max"),
).reset_index()
group_table = group_table.merge(_group_meta, on=GROUP_KEYS)

_n_cross_bucket = (universe.groupby(GROUP_KEYS)["bucket"].nunique() > 1).sum()
print(f"{len(group_table):,} groups  ({_n_cross_bucket:,} span >1 bucket -- assigned their majority "
      f"bucket, ties -> higher)")
print(group_table["n_members"].value_counts().sort_index().rename("groups by member count"))

1,194 groups  (472 span >1 bucket -- assigned their majority bucket, ties -> higher)
n_members
1       1
3    1193
Name: groups by member count, dtype: int64


## Trend + volatility, from pre-treatment weekly revenue

In [4]:
ttd = pd.read_csv(ttd_weekly_file, low_memory=False)
ttd_provider = ttd[ttd["Third_Party_Data_Provider_Id"] == provider]
pre = ttd_provider[ttd_provider["Week_Start_Date"] < treatment_date]

wk_raw = pre.pivot_table(index="Third_Party_Data_Provider_Element_Id", columns="Week_Start_Date",
                          values="Data_Cost", aggfunc="sum", dropna=False)

# auto-detect weeks with too many unmatched element ids (a real data quirk
# found while building this notebook) rather than hand-picking a window.
nan_frac = wk_raw.isna().mean(axis=0)
bad_weeks = nan_frac[nan_frac > max_week_nan_frac].index.tolist()
good_weeks = nan_frac[nan_frac <= max_week_nan_frac].index.tolist()
if bad_weeks:
    print(f"dropping {len(bad_weeks)} week(s) with >{max_week_nan_frac:.0%} unmatched element ids: "
          f"{[w[:10] for w in bad_weeks]}")
print(f"using {len(good_weeks)} clean pre-treatment week(s): {good_weeks[0][:10]} to {good_weeks[-1][:10]}")

wk = wk_raw[good_weeks].fillna(0.0)

dropping 3 week(s) with >50% unmatched element ids: ['2026-01-05', '2026-01-12', '2026-01-19']
using 17 clean pre-treatment week(s): 2026-01-26 to 2026-05-18


In [5]:
x = np.arange(wk.shape[1])
means = wk.mean(axis=1)
stds = wk.std(axis=1)
slopes = wk.apply(lambda row: np.polyfit(x, row.values, 1)[0], axis=1)

trend_rel = (slopes / means.replace(0, np.nan)).abs()
vol_cv = stds / means.replace(0, np.nan)
stability_score = (trend_rel.fillna(np.inf) + vol_cv.fillna(np.inf))

stab_df = pd.DataFrame({
    "Third_Party_Data_Provider_Element_Id": wk.index,
    "stability_score": stability_score.values,
})

universe = universe.merge(stab_df, on="Third_Party_Data_Provider_Element_Id", how="left")
n_no_stability = universe["stability_score"].isna().sum()
universe["stability_score"] = universe["stability_score"].fillna(np.inf)
print(f"{n_no_stability:,} segment(s) have no pre-treatment weekly data -- "
      f"stability_score set to inf (never preferred)")

# propagate to group level: average member stability
_grp_stab = universe.groupby(GROUP_KEYS)["stability_score"].mean().rename("avg_stability").reset_index()
group_table = group_table.merge(_grp_stab, on=GROUP_KEYS)

11 segment(s) have no pre-treatment weekly data -- stability_score set to inf (never preferred)


## Group 1 — holdout (segment-level, at most one per `group`)

In [6]:
def largest_remainder_alloc(shares, total):
    """Apportion `total` integer units across `shares` (a Series of floats
    summing to ~1), largest-remainder method -- so per-bucket sub-targets sum
    exactly to `total`."""
    raw = shares * total
    base = np.floor(raw).astype(int)
    remainder = total - base.sum()
    if remainder > 0:
        order = (raw - base).sort_values(ascending=False).index
        for idx in order[:remainder]:
            base[idx] += 1
    return base


def select_holdout(universe, pct, min_revenue, pct_overrides=None, min_n_overrides=None, min_n_floor=None,
                    candidate_states=None):
    """Segment-level selection, per (bucket, geo) cell: baseline target =
    round(pct * n_geo) allocated across buckets by population share
    (largest-remainder), overridden per-cell by `pct_overrides`/
    `min_n_overrides` exactly like before. `min_n_floor` is a blanket
    minimum applied to every cell not already in `min_n_overrides` (capped
    at that cell's pool size).

    `candidate_states`, if given, is a hard filter -- only segments whose
    `state` is in this list are eligible at all (e.g. ["O", "C1"] excludes
    anything currently under active treatment, T1/T2, from ever being
    nominated as a holdout member, even as a fallback). If a cell's filtered
    pool can't fill its target, that shortfall is reported, not silently
    backfilled from an excluded state -- see the shortfall warning below.
    If None (default), every state is eligible, preferring delta==0 first,
    then lowest stability_score, falling back to delta!=0 candidates only
    when a cell's zero-delta pool runs short.

    Skips any candidate whose `group` has already contributed a selected
    segment (tracked in `used_groups`, shared across every cell in this
    call), so no group appears twice in the holdout."""
    pct_overrides = pct_overrides or {}
    min_n_overrides = min_n_overrides or {}
    eligible = universe[universe["revenue"] > min_revenue].copy()
    if candidate_states is not None:
        n_before = len(eligible)
        eligible = eligible[eligible["state"].isin(candidate_states)]
        print(f"candidate_states={candidate_states} -- {len(eligible):,} of {n_before:,} "
              f"revenue-eligible segment(s) remain eligible")
    print(f"eligible (revenue > {min_revenue}): {len(eligible):,} of {len(universe):,}")

    used_groups = set()
    selected_rows, shortfalls, summary_rows = [], [], []
    for geo in GEO_ORDER:
        geo_pool = eligible[eligible["geo"] == geo]
        n_geo = len(geo_pool)
        if n_geo == 0:
            continue
        target_geo = round(pct * n_geo)
        bucket_counts = geo_pool.groupby("bucket").size().reindex(BUCKET_ORDER, fill_value=0)
        shares = bucket_counts / bucket_counts.sum()
        baseline_targets = largest_remainder_alloc(shares, target_geo)

        for bucket in BUCKET_ORDER:
            cell = geo_pool[geo_pool["bucket"] == bucket]
            if len(cell) == 0:
                summary_rows.append({"geo": geo, "bucket": bucket, "pool_n": 0, "target": 0, "selected": 0})
                continue
            if (bucket, geo) in pct_overrides or (bucket, geo) in min_n_overrides:
                effective_pct = pct_overrides.get((bucket, geo), pct)
                target = round(effective_pct * len(cell))
                if (bucket, geo) in min_n_overrides:
                    target = max(target, min(min_n_overrides[(bucket, geo)], len(cell)))
            else:
                target = int(baseline_targets[bucket])
                if min_n_floor is not None:
                    target = max(target, min(min_n_floor, len(cell)))
            if target == 0:
                summary_rows.append({"geo": geo, "bucket": bucket, "pool_n": len(cell), "target": 0, "selected": 0})
                continue

            ranked = cell.assign(_pref=(cell["delta"] != 0)).sort_values(["_pref", "stability_score"])
            picked = []
            for _, row in ranked.iterrows():
                gkey = (row["Third_Party_Data_Provider_Id"], row["geo"], row["group"])
                if gkey in used_groups:
                    continue
                picked.append(row)
                used_groups.add(gkey)
                if len(picked) == target:
                    break
            if len(picked) < target:
                shortfalls.append({"geo": geo, "bucket": bucket, "target": target, "available": len(picked)})
            selected_rows.append(pd.DataFrame(picked))
            summary_rows.append({"geo": geo, "bucket": bucket, "pool_n": len(cell), "target": target, "selected": len(picked)})

    result = pd.concat(selected_rows, ignore_index=True) if selected_rows else eligible.iloc[0:0].copy()
    return result, pd.DataFrame(summary_rows), shortfalls, used_groups

In [7]:
holdout, g1_summary, g1_shortfalls, g1_used_groups = select_holdout(
    universe, group1_pct, min_revenue, pct_overrides=group1_pct_overrides,
    min_n_overrides=group1_min_n_overrides, min_n_floor=group1_min_n_floor,
    candidate_states=['O', 'C1'])

print(f"\ntotal holdout: {len(holdout):,} segment(s), {len(g1_used_groups):,} distinct groups "
      f"({len(holdout) == len(g1_used_groups)} -- every segment from a different group)")
if g1_shortfalls:
    print(f"WARNING -- {len(g1_shortfalls)} cell(s) had fewer eligible candidates than the target:")
    for s in g1_shortfalls:
        print(f"  {s['geo']} / {s['bucket']}: wanted {s['target']}, only {s['available']} available")
else:
    print("no shortfalls -- every cell had enough eligible candidates")

print("\ndelta composition of holdout segments (their CURRENT delta, unchanged -- holdout means no new delta):")
print(holdout["delta"].value_counts())
print("\n=== selected, by bucket x geo ===")
print(g1_summary.pivot(index="bucket", columns="geo", values="selected").reindex(index=BUCKET_ORDER, columns=GEO_ORDER, fill_value=0))

candidate_states=['O', 'C1'] -- 1,930 of 2,362 revenue-eligible segment(s) remain eligible
eligible (revenue > 0): 1,930 of 3,580

total holdout: 250 segment(s), 250 distinct groups (True -- every segment from a different group)
WARNING -- 5 cell(s) had fewer eligible candidates than the target:
  Asia / 100+: wanted 1, only 0 available
  APAC / 10+: wanted 4, only 2 available
  APAC / 100+: wanted 7, only 1 available
  Unknown / 100+: wanted 10, only 7 available
  Unknown / 300+: wanted 8, only 3 available

delta composition of holdout segments (their CURRENT delta, unchanged -- holdout means no new delta):
delta
0.0    250
Name: count, dtype: int64

=== selected, by bucket x geo ===
geo     Global  US  Asia  APAC  Unknown
bucket                                 
0+          10  25    63     1       15
1+          10  14    10     0       10
10+         10  10    10     2       10
100+        10  10     0     1        7
300+         9  10     0     0        3


In [8]:
holdout_out = holdout.copy()
holdout_out["tag"] = group1_tag + holdout_out["delta"].apply(lambda d: f" (delta = {d:+.3f})")
_cols = ["Third_Party_Data_Provider_Id", "Third_Party_Data_Provider_Element_Id", "Full_Path",
         "geo", "bucket", "group", "revenue", "delta", "stability_score", "tag"]
holdout_out = holdout_out[_cols].sort_values(["geo", "bucket", "stability_score"]).reset_index(drop=True)

if group1_output_file is not None:
    holdout_out.to_csv(group1_output_file, index=False)
    print(f"wrote {len(holdout_out):,} rows to {group1_output_file}")
holdout_out.head(10)

wrote 250 rows to /Users/ravirajan/code/outputs/TTD/holdout_list_2026-07-24.csv


,Third_Party_Data_Provider_Id,Third_Party_Data_Provider_Element_Id,Full_Path,geo,bucket,group,revenue,delta,stability_score,tag
0,sharethis,TTDAPC20,OTP > APAC > Demographic > Gender > Male,APAC,0+,3,0.24,0.0,2.770471,experiment 1 holdout (delta = +0.000)
1,sharethis,TTDAPC29,OTP > APAC > Demographic > Age > Age 50-54,APAC,10+,2,97.10,0.0,0.487780,experiment 1 holdout (delta = +0.000)
2,sharethis,TTDAPC22,OTP > APAC > Demographic > Age > Age 18-20,APAC,10+,0,60.03,0.0,1.073411,experiment 1 holdout (delta = +0.000)
3,sharethis,TTDAPC26,OTP > APAC > Demographic > Age > Age 35-39,APAC,100+,1,270.46,0.0,0.292979,experiment 1 holdout (delta = +0.000)
4,sharethis,32a61e70,Asia > Business & Industrial > Advertising & M...,Asia,0+,73,0.37,0.0,0.148286,experiment 1 holdout (delta = +0.000)
5,sharethis,ffa6ed52,Asia > Business & Industrial > Business Services,Asia,0+,54,0.66,0.0,0.232725,experiment 1 holdout (delta = +0.000)
6,sharethis,8ce6224b,Asia > Business & Industrial > Construction & ...,Asia,0+,124,0.10,0.0,0.245265,experiment 1 holdout (delta = +0.000)
7,sharethis,21965426,Asia > Business & Industrial > Business Servic...,Asia,0+,157,0.04,0.0,0.257694,experiment 1 holdout (delta = +0.000)
8,sharethis,c34b8517,Asia > Business & Industrial > Business Servic...,Asia,0+,119,0.11,0.0,0.263275,experiment 1 holdout (delta = +0.000)
9,sharethis,7084b74a,Asia > Business & Industrial > Advertising & M...,Asia,0+,77,0.43,0.0,0.266549,experiment 1 holdout (delta = +0.000)


In [9]:
# A holdout pick protects its ENTIRE Full_Path (every Element_Id sharing that
# taxonomy node) -- not just the literal Element_Id selected. Some Full_Path
# values span more than one Element_Id (the same data quirk fixed by
# TTD_Price_Assign.ipynb's segment-collapse); without this, Group 2 could
# independently pick that SAME logical segment's other Element_Id, producing
# a segment that's simultaneously "held out" and "an active experiment 2
# candidate" once downstream collapses everything by Full_Path -- exactly
# what happened to "Interest from Social Activity > Seasonal > Summer >
# Sports" before this fix. Holdout wins -- excluded here, before Group 2
# candidates are generated at all. A group's OTHER (different-Full_Path)
# members are still legitimate Group 2 candidates.
holdout_element_ids = set(holdout["Third_Party_Data_Provider_Element_Id"])
holdout_full_paths = set(holdout["Full_Path"])
excluded_full_paths = set(holdout_full_paths)
if exclude_restore_segments_from_group2:
    excluded_full_paths |= set(restore_segments)
    print(f"excluding {len(restore_segments)} restore segment(s) from Group 2 candidates too "
          f"(exclude_restore_segments_from_group2=True)")

# NOTE: compound-Element_Id segments are intentionally NOT excluded here.
# require_unique_element_id_for_treatment is enforced downstream instead, by
# resolve_compound_treatment_violations (after Group 2's RNG-based selection
# runs) -- swapping roles with an in-group control/holdout member so the
# compound segment ends up control or holdout rather than treatment, instead
# of shrinking each affected group's eligible pool upfront (which would just
# shift who else gets randomly selected, for no benefit).
available_universe = universe[~universe["Full_Path"].isin(excluded_full_paths)]

_avail_agg = available_universe.groupby(GROUP_KEYS).agg(
    available_members=("delta", "size"),
    any_nonzero=("delta", lambda s: (s != 0).any()),
    avg_stability=("stability_score", "mean"),
).reset_index()

group_table = group_table.drop(columns=["any_nonzero", "avg_stability"]).merge(
    _avail_agg, on=GROUP_KEYS, how="left")
group_table["available_members"] = group_table["available_members"].fillna(0).astype(int)
group_table["any_nonzero"] = group_table["any_nonzero"].fillna(False)
group_table["avg_stability"] = group_table["avg_stability"].fillna(np.inf)

_n_fully_claimed = (group_table["available_members"] == 0).sum()
print(f"{len(holdout_full_paths):,} Full_Path(s) held out, each from a different group "
      f"({_n_fully_claimed} group(s) fully claimed/excluded -- unavailable to Group 2)")

excluding 10 restore segment(s) from Group 2 candidates too (exclude_restore_segments_from_group2=True)
250 Full_Path(s) held out, each from a different group (4 group(s) fully claimed/excluded -- unavailable to Group 2)


/var/folders/wt/bzrqdlb17zv7h4r1y9jqpg_w0000gp/T/ipykernel_16610/2838382230.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  group_table["any_nonzero"] = group_table["any_nonzero"].fillna(False)


## Group 2 — experiment 2 candidates (group-level selection, random delta assignment)

In [10]:
def select_experiment2_groups(group_table, pct, min_revenue, min_available_members=1,
                               pct_overrides=None, min_n_overrides=None, min_n_floor=None):
    """Same stratified allocation scheme as select_holdout, but the unit of
    selection is a GROUP, not a segment -- `group_table` has one row per
    (provider, geo, group), with `any_nonzero`/`avg_stability` already
    recomputed over each group's AVAILABLE (non-holdout) members only.
    Prefers groups with any_nonzero available member (already touched by the
    current experiment) first, then lowest avg_stability among available
    members. A group stays eligible even if Group 1 claimed one of its
    members -- only fully-claimed groups (available_members < 
    min_available_members) are excluded, so a holdout pick no longer removes
    the rest of its group from Group 2's pool.
    `min_n_floor` is a blanket minimum applied to every (bucket, geo) cell
    that isn't already in `min_n_overrides` (capped at that cell's pool
    size) -- `min_n_overrides` still wins for the cells it names."""
    pct_overrides = pct_overrides or {}
    min_n_overrides = min_n_overrides or {}

    eligible = group_table[
        (group_table["max_revenue"] > min_revenue) &
        (group_table["available_members"] >= min_available_members)
    ].copy()
    print(f"eligible groups (max_revenue > {min_revenue}, available_members >= {min_available_members}): "
          f"{len(eligible):,} of {len(group_table):,}")

    selected_rows, shortfalls, summary_rows = [], [], []
    for geo in GEO_ORDER:
        geo_pool = eligible[eligible["geo"] == geo]
        n_geo = len(geo_pool)
        if n_geo == 0:
            continue
        target_geo = round(pct * n_geo)
        bucket_counts = geo_pool.groupby("bucket").size().reindex(BUCKET_ORDER, fill_value=0)
        shares = bucket_counts / bucket_counts.sum()
        baseline_targets = largest_remainder_alloc(shares, target_geo)

        for bucket in BUCKET_ORDER:
            cell = geo_pool[geo_pool["bucket"] == bucket]
            if len(cell) == 0:
                summary_rows.append({"geo": geo, "bucket": bucket, "pool_n": 0, "target": 0, "selected": 0})
                continue
            if (bucket, geo) in pct_overrides or (bucket, geo) in min_n_overrides:
                effective_pct = pct_overrides.get((bucket, geo), pct)
                target = round(effective_pct * len(cell))
                if (bucket, geo) in min_n_overrides:
                    target = max(target, min(min_n_overrides[(bucket, geo)], len(cell)))
            else:
                target = int(baseline_targets[bucket])
                if min_n_floor is not None:
                    target = max(target, min(min_n_floor, len(cell)))
            if target == 0:
                summary_rows.append({"geo": geo, "bucket": bucket, "pool_n": len(cell), "target": 0, "selected": 0})
                continue

            ranked = cell.assign(_pref=~cell["any_nonzero"]).sort_values(["_pref", "avg_stability"])
            picked = ranked.head(target)
            if len(picked) < target:
                shortfalls.append({"geo": geo, "bucket": bucket, "target": target, "available": len(picked)})
            selected_rows.append(picked)
            summary_rows.append({"geo": geo, "bucket": bucket, "pool_n": len(cell), "target": target, "selected": len(picked)})

    result = pd.concat(selected_rows, ignore_index=True) if selected_rows else eligible.iloc[0:0].copy()
    return result, pd.DataFrame(summary_rows), shortfalls

In [11]:
sel_groups, g2_summary, g2_shortfalls = select_experiment2_groups(
    group_table, group2_pct, min_revenue,
    pct_overrides=group2_pct_overrides, min_n_overrides=group2_min_n_overrides,
    min_n_floor=group2_min_n_floor)

print(f"\ntotal groups selected: {len(sel_groups):,}")
if g2_shortfalls:
    print(f"WARNING -- {len(g2_shortfalls)} cell(s) had fewer eligible groups than the target:")
    for s in g2_shortfalls:
        print(f"  {s['geo']} / {s['bucket']}: wanted {s['target']}, only {s['available']} available")
else:
    print("no shortfalls -- every cell had enough eligible groups")

print("\n=== groups selected, by bucket x geo ===")
print(g2_summary.pivot(index="bucket", columns="geo", values="selected").reindex(index=BUCKET_ORDER, columns=GEO_ORDER, fill_value=0))
print("\n=== eligible group pool, by bucket x geo (for reference) ===")
print(g2_summary.pivot(index="bucket", columns="geo", values="pool_n").reindex(index=BUCKET_ORDER, columns=GEO_ORDER, fill_value=0))

eligible groups (max_revenue > 0, available_members >= 1): 1,027 of 1,194

total groups selected: 143
no shortfalls -- every cell had enough eligible groups

=== groups selected, by bucket x geo ===
geo     Global  US  Asia  APAC  Unknown
bucket                                 
0+          14  13    33     0       10
1+           5   5     5     0        5
10+          7   5     5     1        5
100+         5   6     0     3        3
300+         3   5     0     0        5

=== eligible group pool, by bucket x geo (for reference) ===
geo     Global   US  Asia  APAC  Unknown
bucket                                  
0+         137  131   334     0       95
1+          46   40     9     0       21
10+         74   32    10     1       23
100+        17   19     0     3        3
300+         9   18     0     0        5


In [12]:
# randomly assign one of NEW_DELTAS to each selected group's AVAILABLE
# (non-holdout) members only -- a holdout pick keeps its own segment
# untouched, but its group's other members are still fair game. Groups with
# != 3 available members (rare) get as many of NEW_DELTAS as they have
# available members (no repeats within a group); any member beyond
# len(NEW_DELTAS) is left out of this round's assignment.
#
# Each group draws from its OWN independent RNG, seeded from a stable hash of
# its (provider, geo, group) identity -- not sequential draws from one shared
# RNG stream. A shared stream means group N's random draw depends on exactly
# how many draws groups 1..N-1 consumed, which depends on which OTHER groups
# got selected upstream (e.g. by Group 1/holdout selection) -- so an
# unrelated change elsewhere in the pipeline can silently reshuffle every
# later group's assignment (confirmed: a Group-1 selection-logic fix once
# changed 224 unrelated Group-2 assignments this way). Per-group hashed seeds
# make a group's own assignment depend only on its own identity and
# NEW_DELTAS/member set -- stable against any upstream reshuffling.
def _stable_group_seed(key):
    key_str = "|".join(str(k) for k in key)
    digest = hashlib.sha256(key_str.encode()).hexdigest()
    return int(digest[:16], 16) % (2**32)

assign_rows = []
for _, g in sel_groups.iterrows():
    members = universe[
        (universe["Third_Party_Data_Provider_Id"] == g["Third_Party_Data_Provider_Id"]) &
        (universe["geo"] == g["geo"]) & (universe["group"] == g["group"]) &
        (~universe["Full_Path"].isin(excluded_full_paths))
    ]
    k = min(len(members), len(NEW_DELTAS))
    gkey = (g["Third_Party_Data_Provider_Id"], g["geo"], g["group"])
    group_rng = np.random.default_rng(_stable_group_seed(gkey))
    chosen_idx = group_rng.choice(members.index, size=k, replace=False)
    chosen_deltas = group_rng.permutation(NEW_DELTAS)[:k]
    for idx, d in zip(chosen_idx, chosen_deltas):
        row = members.loc[idx].copy()
        row["new_delta"] = d
        assign_rows.append(row)

exp2_out = pd.DataFrame(assign_rows)
exp2_out["tag"] = (np.where(exp2_out["new_delta"] == 0, group2_control_tag, group2_tag)
                   + exp2_out["new_delta"].apply(lambda d: f" (delta = {d:+.3f})"))
print(f"{len(exp2_out):,} segment(s) assigned across {len(sel_groups):,} groups")
print(exp2_out["new_delta"].value_counts())

_cols = ["Third_Party_Data_Provider_Id", "Third_Party_Data_Provider_Element_Id", "Full_Path",
         "geo", "bucket", "group", "revenue", "delta", "new_delta", "stability_score", "tag"]
exp2_out = exp2_out[_cols].sort_values(["geo", "bucket", "group"]).reset_index(drop=True)

if group2_output_file is not None:
    exp2_out.to_csv(group2_output_file, index=False)
    print(f"wrote {len(exp2_out):,} rows to {group2_output_file}")
exp2_out.head(10)

337 segment(s) assigned across 143 groups
new_delta
-0.025    117
 0.025    113
 0.000    107
Name: count, dtype: int64
wrote 337 rows to /Users/ravirajan/code/outputs/TTD/experiment2_candidates_2026-07-24.csv


,Third_Party_Data_Provider_Id,Third_Party_Data_Provider_Element_Id,Full_Path,geo,bucket,group,revenue,delta,new_delta,stability_score,tag
0,sharethis,TTDAPC31,OTP > APAC > Demographic > Age > Age 65+,APAC,10+,3,48.20,0.0,0.000,0.959659,control 2 (delta = +0.000)
1,sharethis,TTDAPC21,OTP > APAC > Demographic > Gender > Female,APAC,10+,3,29.42,0.0,0.025,1.104599,experiment 2 (delta = +0.025)
2,sharethis,TTDAPC23,OTP > APAC > Demographic > Age > Age 21-24,APAC,100+,0,116.26,0.0,-0.025,1.039184,experiment 2 (delta = -0.025)
3,sharethis,TTDAPC30,OTP > APAC > Demographic > Age > Age 55-64,APAC,100+,0,112.88,0.0,0.025,0.754360,experiment 2 (delta = +0.025)
4,sharethis,TTDAPC27,OTP > APAC > Demographic > Age > Age 40-44,APAC,100+,1,139.46,0.0,-0.025,0.537744,experiment 2 (delta = -0.025)
5,sharethis,TTDAPC24,OTP > APAC > Demographic > Age > Age 25-29,APAC,100+,1,248.66,0.0,0.025,0.352748,experiment 2 (delta = +0.025)
6,sharethis,TTDAPC28,OTP > APAC > Demographic > Age > Age 45-49,APAC,100+,2,106.49,0.0,0.025,0.494254,experiment 2 (delta = +0.025)
7,sharethis,TTDAPC25,OTP > APAC > Demographic > Age > Age 30-34,APAC,100+,2,201.28,0.0,0.000,0.210526,control 2 (delta = +0.000)
8,sharethis,4910200a,Asia > The Changing Consumer > Gaming Equipment,Asia,0+,7,0.61,0.0,0.025,1.997626,experiment 2 (delta = +0.025)
9,sharethis,92233f54,Asia > Home & Garden,Asia,0+,8,0.00,0.0,0.000,1.955227,control 2 (delta = +0.000)


In [13]:
# require_unique_element_id_for_treatment forbids a compound-Element_Id
# segment (Full_Path spans >1 raw element, delimited with "|") from holding a
# TREATMENT role -- pricing such a segment as a single unit is ambiguous,
# since it's really 2+ distinct elements. Any violation gets resolved within
# its own group, trying each option in order:
#   1. swap roles with an existing, non-compound CONTROL member of the same
#      group (control <-> treatment) -- e.g. "Summer > Shoppers" (compound,
#      treatment) traded roles with "Grains & Pasta" (control) in the same
#      group, 2026-07-24 round.
#   2. else swap roles with an existing, non-compound HOLDOUT member of the
#      same group -- that member is promoted to treatment (taking over the
#      compound segment's new_delta), the compound segment is demoted to
#      holdout instead. e.g. "Summer > Sports" (compound, treatment) traded
#      with "B2B > Industry > Food Services" (holdout) when its own group
#      had no control member to use for step 1.
#   3. else drop the compound segment's treatment assignment entirely
#      (removed from exp2_out, not replaced) -- logged as a shortfall.
# Re-scans after every resolution (not just once per violation found
# up-front): a swap changes which rows are violations/control/holdout
# candidates, so a group with more than one violation, or a chain where step
# 2's promoted member could itself free up a step-1 opportunity for another
# violation, is still handled correctly.
def resolve_compound_treatment_violations(universe, holdout_out, exp2_out,
                                           group1_tag, group2_tag, group2_control_tag):
    holdout_out = holdout_out.reset_index(drop=True).copy()
    exp2_out = exp2_out.reset_index(drop=True).copy()
    log = []

    def is_compound(eid_series):
        return eid_series.str.contains("|", regex=False)

    _carry_cols = ["Third_Party_Data_Provider_Id", "Third_Party_Data_Provider_Element_Id",
                   "Full_Path", "geo", "bucket", "group", "revenue", "delta", "stability_score"]

    while True:
        viol_mask = is_compound(exp2_out["Third_Party_Data_Provider_Element_Id"]) & (exp2_out["new_delta"] != 0)
        if not viol_mask.any():
            break
        v_idx = exp2_out.index[viol_mask][0]
        v = exp2_out.loc[v_idx]
        gkey = (v["Third_Party_Data_Provider_Id"], v["geo"], v["group"])
        group_paths = set(universe[
            (universe["Third_Party_Data_Provider_Id"] == gkey[0]) &
            (universe["geo"] == gkey[1]) & (universe["group"] == gkey[2])
        ]["Full_Path"]) - {v["Full_Path"]}

        # Step 1: existing control (new_delta==0), non-compound, same group
        control_cands = exp2_out[
            exp2_out["Full_Path"].isin(group_paths) & (exp2_out["new_delta"] == 0) &
            ~is_compound(exp2_out["Third_Party_Data_Provider_Element_Id"])
        ]
        if len(control_cands):
            c_idx = control_cands.index[0]
            orig_delta = v["new_delta"]
            exp2_out.loc[c_idx, "new_delta"] = orig_delta
            exp2_out.loc[c_idx, "tag"] = f"{group2_tag} (delta = {orig_delta:+.3f})"
            exp2_out.loc[v_idx, "new_delta"] = 0.0
            exp2_out.loc[v_idx, "tag"] = f"{group2_control_tag} (delta = +0.000)"
            log.append({"violation": v["Full_Path"], "resolution": "step1_swap_control",
                        "swapped_with": exp2_out.loc[c_idx, "Full_Path"]})
            continue

        # Step 2: existing holdout, non-compound, same group
        holdout_cands = holdout_out[
            holdout_out["Full_Path"].isin(group_paths) &
            ~is_compound(holdout_out["Third_Party_Data_Provider_Element_Id"])
        ]
        if len(holdout_cands):
            h_idx = holdout_cands.index[0]
            h_row = holdout_out.loc[h_idx]
            orig_delta = v["new_delta"]

            new_exp2_row = h_row[_carry_cols].to_dict()
            new_exp2_row["new_delta"] = orig_delta
            new_exp2_row["tag"] = f"{group2_tag} (delta = {orig_delta:+.3f})"

            new_holdout_row = v[_carry_cols].to_dict()
            new_holdout_row["tag"] = f"{group1_tag} (delta = {new_holdout_row['delta']:+.3f})"

            exp2_out = exp2_out.drop(index=v_idx)
            exp2_out = pd.concat([exp2_out, pd.DataFrame([new_exp2_row])], ignore_index=True)
            holdout_out = holdout_out.drop(index=h_idx)
            holdout_out = pd.concat([holdout_out, pd.DataFrame([new_holdout_row])], ignore_index=True)

            log.append({"violation": v["Full_Path"], "resolution": "step2_swap_holdout",
                        "swapped_with": h_row["Full_Path"]})
            continue

        # Step 3: drop
        exp2_out = exp2_out.drop(index=v_idx)
        log.append({"violation": v["Full_Path"], "resolution": "step3_dropped", "swapped_with": None})

    return holdout_out, exp2_out, log


if require_unique_element_id_for_treatment:
    holdout_out, exp2_out, _resolution_log = resolve_compound_treatment_violations(
        universe, holdout_out, exp2_out, group1_tag, group2_tag, group2_control_tag)

    print(f"{len(_resolution_log)} compound-Element_Id treatment violation(s) found and resolved:")
    for entry in _resolution_log:
        print(f"  {entry['resolution']:20s} {entry['violation']!r}"
              + (f"  <->  {entry['swapped_with']!r}" if entry["swapped_with"] else "  (no replacement -- dropped)"))
    if not _resolution_log:
        print("  none")

    _cols_h = ["Third_Party_Data_Provider_Id", "Third_Party_Data_Provider_Element_Id", "Full_Path",
               "geo", "bucket", "group", "revenue", "delta", "stability_score", "tag"]
    _cols_e = ["Third_Party_Data_Provider_Id", "Third_Party_Data_Provider_Element_Id", "Full_Path",
               "geo", "bucket", "group", "revenue", "delta", "new_delta", "stability_score", "tag"]
    holdout_out = holdout_out[_cols_h].sort_values(["geo", "bucket", "stability_score"]).reset_index(drop=True)
    exp2_out = exp2_out[_cols_e].sort_values(["geo", "bucket", "group"]).reset_index(drop=True)

    if group1_output_file is not None:
        holdout_out.to_csv(group1_output_file, index=False)
        print(f"\nre-wrote {len(holdout_out):,} rows to {group1_output_file} (post-resolution)")
    if group2_output_file is not None:
        exp2_out.to_csv(group2_output_file, index=False)
        print(f"re-wrote {len(exp2_out):,} rows to {group2_output_file} (post-resolution)")
else:
    print("require_unique_element_id_for_treatment is False -- skipping compound-Element_Id resolution")


1 compound-Element_Id treatment violation(s) found and resolved:
  step1_swap_control   'Interest from Social Activity > Seasonal > Summer > Travelers'  <->  'Intent > Auto Buyers > New'



re-wrote 250 rows to /Users/ravirajan/code/outputs/TTD/holdout_list_2026-07-24.csv (post-resolution)
re-wrote 337 rows to /Users/ravirajan/code/outputs/TTD/experiment2_candidates_2026-07-24.csv (post-resolution)


In [14]:
# Groups CAN legitimately appear in both lists now (holdout claims one
# member, Group 2 reassigns the rest) -- what must never happen is the same
# LOGICAL SEGMENT (Full_Path) appearing in both. All checks below use the
# FINAL holdout_out/exp2_out (post compound-treatment resolution above) --
# the pre-resolution `holdout`/holdout_full_paths are stale once that step
# has possibly moved segments between the two files.
print("=== Diagnostics ===")

group_overlap = set(zip(holdout_out["Third_Party_Data_Provider_Id"], holdout_out["geo"], holdout_out["group"])) & \
                set(zip(exp2_out["Third_Party_Data_Provider_Id"], exp2_out["geo"], exp2_out["group"]))
print(f"groups appearing in both lists: {len(group_overlap)} (expected -- shared groups, different members)")

element_overlap = set(holdout_out["Third_Party_Data_Provider_Element_Id"]) & set(exp2_out["Third_Party_Data_Provider_Element_Id"])
print(f"Element_Id appearing in both lists: {len(element_overlap)} (should be 0)")

# The real invariant, and the one raw Element_Id overlap CANNOT catch: no
# Full_Path (the authoritative segment identity downstream) may appear in
# both lists, even via a different Element_Id of the same taxonomy node --
# this is exactly the class of bug that let "Summer > Sports" end up in both
# lists despite 0 raw Element_Id overlap (its two Element_Ids were split
# across holdout and Group 2).
final_holdout_paths = set(holdout_out["Full_Path"])
full_path_overlap = final_holdout_paths & set(exp2_out["Full_Path"])
if full_path_overlap:
    print(f"WARNING: {len(full_path_overlap)} Full_Path(s) appear in BOTH holdout and experiment 2:")
    for fp in sorted(full_path_overlap):
        print(f"  {fp}")
else:
    print("Full_Path appearing in both lists: 0 (holdout correctly wins over experiment-2 candidacy)")

restore_overlap = set(restore_segments) & set(exp2_out["Full_Path"])
if exclude_restore_segments_from_group2:
    print(f"restore_segments appearing in experiment 2 candidates: {len(restore_overlap)} "
          f"(should be 0 -- exclude_restore_segments_from_group2=True)")
else:
    print(f"restore_segments appearing in experiment 2 candidates: {len(restore_overlap)} "
          f"(exclude_restore_segments_from_group2=False -- not excluded, informational only)")

# A compound-Element_Id segment holding CONTROL (new_delta==0) is fine -- no
# differential price is being applied, so the "which element does this
# really price" ambiguity never bites. Only compound + nonzero new_delta
# (an actual TREATMENT assignment) is the real violation.
compound_mask = exp2_out["Third_Party_Data_Provider_Element_Id"].str.contains("|", regex=False)
compound_treatment_violations = exp2_out[compound_mask & (exp2_out["new_delta"] != 0)]
if require_unique_element_id_for_treatment:
    print(f"compound-Element_Id segments holding a TREATMENT role: {len(compound_treatment_violations)} "
          f"(should be 0 -- require_unique_element_id_for_treatment=True)")
else:
    print(f"compound-Element_Id segments holding a TREATMENT role: {len(compound_treatment_violations)} "
          f"(require_unique_element_id_for_treatment=False -- not resolved, informational only)")

# Sanity: neither output file should contain the same Full_Path twice
# internally either (a group-selection bug distinct from the holdout/exp2
# overlap above).
dup_in_holdout = holdout_out[holdout_out.duplicated(subset=["Third_Party_Data_Provider_Id", "Full_Path"], keep=False)]
print(f"duplicate Full_Path within the holdout list itself: {len(dup_in_holdout)} rows (should be 0)")

dup_in_exp2 = exp2_out[exp2_out.duplicated(subset=["Third_Party_Data_Provider_Id", "Full_Path"], keep=False)]
print(f"duplicate Full_Path within the experiment 2 list itself: {len(dup_in_exp2)} rows (should be 0)")

=== Diagnostics ===
groups appearing in both lists: 92 (expected -- shared groups, different members)
Element_Id appearing in both lists: 0 (should be 0)
Full_Path appearing in both lists: 0 (holdout correctly wins over experiment-2 candidacy)
restore_segments appearing in experiment 2 candidates: 0 (should be 0 -- exclude_restore_segments_from_group2=True)
compound-Element_Id segments holding a TREATMENT role: 0 (should be 0 -- require_unique_element_id_for_treatment=True)
duplicate Full_Path within the holdout list itself: 0 rows (should be 0)
duplicate Full_Path within the experiment 2 list itself: 0 rows (should be 0)
